# Census ACS 5-Year Raw Data Quality

DP02, DP03, DP04, and S2503 county estimates for 2010–2024.

The audit queries the current `raw.*` ingestion tables directly; transformed tables are out of scope.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "housing_predict.duckdb"), read_only=True)
TABLES = ["dp02","dp03","dp04","s2503"]
KEYS = {"dp02":["year","county_fips"],"dp03":["year","county_fips"],"dp04":["year","county_fips"],"s2503":["year","county_fips"]}
YEAR_COLUMNS = {"dp02":"year","dp03":"year","dp04":"year","s2503":"year"}
EXPECTED_YEARS = set(range(2010, 2025))
def qi(x): return '"' + x.replace('"', '""') + '"'
def columns(t):
    return con.execute("SELECT column_name FROM information_schema.columns WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [t]).df()["column_name"].tolist()
available = set(con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"])
assert set(TABLES) <= available, f"Missing raw tables: {set(TABLES)-available}"


## Inventory, coverage, keys, and sampled missingness

In [ ]:
inventory, key_checks, coverage, missing = [], [], [], []
for table in TABLES:
    cols = columns(table)
    n = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    inventory.append({"table": table, "rows": n, "columns": len(cols)})
    key = KEYS[table]
    key_sql = ", ".join(qi(c) for c in key)
    invalid = " OR ".join(f"{qi(c)} IS NULL OR trim(cast({qi(c)} AS VARCHAR))=''" for c in key)
    key_checks.append({
        "table": table, "candidate_key": " + ".join(key),
        "invalid_key_rows": con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}").fetchone()[0],
        "duplicate_key_groups": con.execute(f"SELECT count(*) FROM (SELECT {key_sql}, count(*) n FROM raw.{qi(table)} GROUP BY {key_sql} HAVING n>1)").fetchone()[0],
    })
    year_col = YEAR_COLUMNS.get(table)
    if year_col:
        years = set(con.execute(f"SELECT DISTINCT try_cast({qi(year_col)} AS INTEGER) FROM raw.{qi(table)} WHERE try_cast({qi(year_col)} AS INTEGER) IS NOT NULL").df().iloc[:,0])
        coverage.append({"table":table, "min_year":min(years), "max_year":max(years), "missing_years":sorted(EXPECTED_YEARS-years), "outside_scope":sorted(years-EXPECTED_YEARS)})
    # Identifier fields plus a bounded sample of wide value fields keeps the audit runnable.
    profile_cols = list(dict.fromkeys([*key, *(["NAME"] if "NAME" in cols else []), *cols[:40]]))
    for col in profile_cols:
        m = con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {qi(col)} IS NULL OR trim(cast({qi(col)} AS VARCHAR))=''").fetchone()[0]
        if m: missing.append({"table":table,"column":col,"missing_rows":m,"missing_pct":round(100*m/n,2)})
display(pd.DataFrame(inventory))
display(pd.DataFrame(coverage))
display(pd.DataFrame(key_checks))
display(pd.DataFrame(missing).sort_values("missing_pct",ascending=False).head(100) if missing else pd.DataFrame())


## Source-specific checks

In [ ]:
display(con.execute("""
SELECT try_cast(year AS INTEGER) AS year, count(*) AS row_count,
 count(DISTINCT county_fips) AS counties,
 count(try_cast(CASE WHEN try_cast(year AS INTEGER)<=2014 THEN DP04_0088E ELSE DP04_0089E END AS DOUBLE)) AS usable_target,
 min(try_cast(CASE WHEN try_cast(year AS INTEGER)<=2014 THEN DP04_0088E ELSE DP04_0089E END AS DOUBLE)) AS min_target,
 max(try_cast(CASE WHEN try_cast(year AS INTEGER)<=2014 THEN DP04_0088E ELSE DP04_0089E END AS DOUBLE)) AS max_target
FROM raw.dp04 GROUP BY 1 ORDER BY 1
""").df())
display(con.execute("""
SELECT 'dp02' AS product, count(*) FILTER (WHERE NOT regexp_matches(county_fips,'^[0-9]{5}$')) AS invalid_fips FROM raw.dp02
UNION ALL SELECT 'dp03', count(*) FILTER (WHERE NOT regexp_matches(county_fips,'^[0-9]{5}$')) FROM raw.dp03
UNION ALL SELECT 'dp04', count(*) FILTER (WHERE NOT regexp_matches(county_fips,'^[0-9]{5}$')) FROM raw.dp04
UNION ALL SELECT 's2503', count(*) FILTER (WHERE NOT regexp_matches(county_fips,'^[0-9]{5}$')) FROM raw.s2503
""").df())

## Interpretation and limitations

Five-year values represent overlapping rolling periods. Preserve margins of error and release-year provenance. The DP04 target field changes from `DP04_0088E` to `DP04_0089E` in 2015.

Displayed failures are investigation items. This notebook does not alter source data.

In [ ]:
con.close()